# Multi-Agent Systems in LangChain v1

This notebook introduces the five official multi-agent patterns in LangChain v1, explains
when to use each one, and shows a minimal runnable example of each. The four notebooks that
follow this one each build a complete, realistic project around one pattern:

| # | Notebook | Pattern | Project |
|---|---|---|---|
| 1 | `01_Subagents_Personal_Assistant.ipynb` | Subagents | A personal assistant with a calendar sub-agent and an email sub-agent |
| 2 | `02_Handoffs_Customer_Support.ipynb` | Handoffs | A customer support agent that changes behavior as it moves through a workflow |
| 3 | `03_Router_Knowledge_Base.ipynb` | Router | A knowledge base that queries GitHub, Notion, and Slack in parallel |
| 4 | `04_Skills_SQL_Assistant.ipynb` | Skills | A SQL assistant that loads database schemas on demand |

A fifth pattern, **Custom Workflow**, is a bespoke LangGraph graph that can embed any of the
other four -- it has no dedicated project notebook here because it is really "build your own
graph," and the Router project notebook already shows a hand-built `StateGraph` in practice.

## Why multi-agent at all?

A single agent with every tool and every instruction mixed into one system prompt works fine
until it doesn't: the model has to choose between dozens of similar-looking tools, hold
unrelated domains of knowledge in one context window, and follow one prompt that's trying to
cover every situation at once. Multi-agent architectures split that single agent into focused
pieces, each with a narrow job, and compose them back together.

```mermaid
graph LR
    subgraph "Single agent"
        A1[One agent] --> T1[20+ tools,<br/>one long prompt]
    end
    subgraph "Multi-agent"
        A2[Coordinator] --> B1[Specialist A<br/>few tools]
        A2 --> B2[Specialist B<br/>few tools]
        A2 --> B3[Specialist C<br/>few tools]
    end
```

The core idea in LangChain v1 is: **`Agent = Model + Harness`**. A multi-agent system is just
several of these Agent objects, wired together in one of a small number of well-understood
shapes. Learning the five shapes below means you can recognize (and build) almost any
multi-agent system you'll encounter.

## Setup

This notebook (and the four that follow) need a real OpenAI API key, since they run live
model calls throughout -- there is no scripted fallback in the student-facing versions. Copy
`.env.example` to `.env` in this folder and set `OPENAI_API_KEY=sk-...` before running.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Copy .env.example to .env in this folder and add your key."
    )

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-mini")
print("Model ready.")


## The five patterns at a glance

| Pattern | Core mechanism | Best fit |
|---|---|---|
| **Subagents** | A supervisor agent calls other agents as tools | Distinct domains, each needing its own tools and prompt, with centralized control |
| **Handoffs** | One agent's prompt and tools change based on state | A sequential workflow where behavior should change as it progresses (or several agents trade off via `Command`) |
| **Skills** | One agent loads specialized prompt content on demand | Many possible specializations, loaded only when needed, to keep context small |
| **Router** | A classification step fans out to specialized agents in parallel, then synthesizes | Distinct knowledge *verticals* that can be queried independently and combined |
| **Custom Workflow** | A hand-built LangGraph graph, any shape | Anything the other four don't fit cleanly, or a mix of several patterns |

```mermaid
graph TD
    Q{What does your system need?}
    Q -->|Centralized control over<br/>distinct domains| SA[Subagents]
    Q -->|Behavior changes as a<br/>task progresses| HO[Handoffs]
    Q -->|Many specializations,<br/>load only what's needed| SK[Skills]
    Q -->|Independent sources,<br/>query in parallel| RO[Router]
    Q -->|None of the above fit| CW[Custom Workflow]
```

### A quick distinction: Subagents vs. Handoffs vs. Router

These three are the ones most often confused with each other, so it's worth being precise
about what separates them:

- **Subagents**: the *supervisor* is itself a full agent. It decides, turn by turn, which
  sub-agent-as-tool to call, and can call several in sequence or in parallel. The sub-agents
  don't talk to the user directly.
- **Handoffs**: there's no supervisor doing live reasoning about where to route. Instead, state
  (such as `current_step`, or which agent is "active") determines which prompt and tools are in
  effect right now. Agents can hand off to each other, or a single agent can reconfigure itself.
- **Router**: a *classification step* (often one structured-output LLM call, not a full agent)
  decides, once, which specialized agents are relevant, dispatches to all of them **in
  parallel**, and a synthesis step combines their answers. There's no multi-turn orchestration.

```mermaid
graph LR
    subgraph Subagents
        S1[Supervisor agent] -->|tool call| S2[Sub-agent A]
        S1 -->|tool call| S3[Sub-agent B]
        S2 -.->|returns text| S1
        S3 -.->|returns text| S1
    end


In [ ]:
# Pattern 1: Subagents -- minimal example.
# A supervisor calls specialist agents as tools. The supervisor never sees the specialists'
# internal reasoning or tool calls -- only their final answers.
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def check_seat_availability(showtime: str) -> str:
    """Check how many seats are left for a showtime."""
    return f"{showtime}: 12 seats remaining in Screen 3."

booking_specialist = create_agent(
    model,
    tools=[check_seat_availability],
    system_prompt="You are a booking specialist. Use your tool to answer seat questions.",
)

@tool
def handle_booking_question(request: str) -> str:
    """Delegate a booking-related question to the booking specialist."""
    result = booking_specialist.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content

supervisor = create_agent(
    model,
    tools=[handle_booking_question],
    system_prompt="You are a helpful front-of-house assistant. Delegate booking questions.",
)

result = supervisor.invoke({"messages": [{"role": "user", "content": "How many seats are left for the 7pm Interstellar showing?"}]})
print(result["messages"][-1].content)


```mermaid
graph TD
    subgraph Handoffs
        H1[Agent, config A] -->|state changes| H2[Same agent, config B]
    end
```


In [ ]:
# Pattern 2: Handoffs -- minimal example.
# A single agent's prompt and tools change based on a state field, via middleware.
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing_extensions import NotRequired
from typing import Literal, Callable

class TicketState(AgentState):
    stage: NotRequired[Literal["greeting", "booking"]]

@wrap_model_call
def apply_stage(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    stage = request.state.get("stage", "greeting")
    if stage == "greeting":
        request = request.override(system_prompt="Greet the customer and ask what movie they want.")
    else:
        request = request.override(system_prompt="Help the customer pick a showtime and confirm the booking.")
    return handler(request)

handoff_agent = create_agent(model, tools=[], middleware=[apply_stage], state_schema=TicketState)
result = handoff_agent.invoke({"messages": [{"role": "user", "content": "Hi"}], "stage": "greeting"})
print(result["messages"][-1].content)


```mermaid
graph LR
    subgraph Skills
        K1[Agent] -->|load_skill tool call| K2[Full skill content<br/>enters context]
        K2 --> K1
    end
```


In [ ]:
# Pattern 3: Skills -- minimal example.
# The agent sees lightweight skill descriptions up front, and loads full content on demand.
SKILLS = {
    "refund_policy": "Full refund if cancelled 2+ hours before showtime; 50% credit within 2 hours.",
}

@tool
def load_skill(skill_name: str) -> str:
    """Load a specialized skill's full content.

    Available skills:
    - refund_policy: refund and cancellation rules
    """
    return SKILLS.get(skill_name, f"Unknown skill: {skill_name}")

skills_agent = create_agent(
    model,
    tools=[load_skill],
    system_prompt="You are a CineBot assistant. Use load_skill when a question needs policy detail.",
)
result = skills_agent.invoke({"messages": [{"role": "user", "content": "If I cancel 3 hours before the show, do I get a refund?"}]})
print(result["messages"][-1].content)


```mermaid
graph LR
    subgraph Router
        R1([Query]) --> R2[Classify]
        R2 --> R3[Agent A]
        R2 --> R4[Agent B]
        R3 --> R5[Synthesize]
        R4 --> R5
    end
```


In [ ]:
# Pattern 4: Router -- minimal example.
# A classification step fans out to agents in parallel, then a synthesis step combines results.
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class RouterState(TypedDict, total=False):
    query: str
    answers: Annotated[list[str], operator.add]
    final: str

def classify(query: str) -> list[str]:
    targets = []
    if "showtime" in query.lower() or "time" in query.lower():
        targets.append("showtimes")
    if "refund" in query.lower():
        targets.append("policy")
    return targets or ["showtimes"]

def route(state: RouterState):
    return [Send(t, {"query": state["query"]}) for t in classify(state["query"])]

def showtimes_node(state) -> dict:
    return {"answers": ["Interstellar: 7:00 PM and 10:15 PM."]}

def policy_node(state) -> dict:
    return {"answers": ["Refunds available up to 2 hours before showtime."]}

def synthesize(state: RouterState) -> dict:
    return {"final": " | ".join(state["answers"])}

router_graph = (
    StateGraph(RouterState)
    .add_node("showtimes", showtimes_node)
    .add_node("policy", policy_node)
    .add_node("synthesize", synthesize)
    .add_conditional_edges(START, route, ["showtimes", "policy"])
    .add_edge("showtimes", "synthesize")
    .add_edge("policy", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)
result = router_graph.invoke({"query": "What time does it show and can I get a refund?"})
print(result["final"])


```mermaid
graph TD
    subgraph "Custom Workflow"
        C1([Input]) --> C2{Decide}
        C2 -->|path A| C3[Deterministic step]
        C2 -->|path B| C4((Agent step))
        C3 --> C5([Output])
        C4 --> C5
    end
```

A custom workflow is just a `StateGraph` you design by hand, node by node. The Router example
above is already a custom workflow -- there's no separate "custom workflow API," it's the same
`StateGraph`/`START`/`END` building blocks used in every pattern above, arranged however your
problem needs. Nothing further to add here; the Router project notebook that follows is a full
worked example of hand-building a graph like this for a real use case.

## Choosing a pattern

| Your situation | Pattern |
|---|---|
| A central agent should keep conversational control, calling out to specialists | Subagents |
| One task flows through clear stages, each needing different tools/instructions | Handoffs |
| Many optional specializations, most irrelevant to any given request | Skills |
| Independent knowledge sources that can be queried at the same time | Router |
| None of the above describe your system exactly | Custom Workflow |

Patterns combine freely. A Subagents system can have a sub-agent that's internally a Router.
A Handoffs stage can use Skills to manage its own context. Pick per sub-problem, not once for
the whole system.

## What's next

Each of the next four notebooks builds one of these patterns into a complete, realistic
project, straight from LangChain's own documentation, with the full set of diagrams,
architecture notes, and working code for that project:

1. `01_Subagents_Personal_Assistant.ipynb`
2. `02_Handoffs_Customer_Support.ipynb`
3. `03_Router_Knowledge_Base.ipynb`
4. `04_Skills_SQL_Assistant.ipynb`
